# Generate Console Tools Docs
This file traverses the console tools folder, executes each tool with the `--help` command, and captures the stdout.
The stdout is then added to the `docs/` folder to auto-generate documentation.

In [1]:
import sys, os
import glob
import subprocess
from tqdm import tqdm
from io import StringIO
from contextlib import redirect_stdout
from pprint import pprint

In [2]:
console_tools_paths = glob.glob(os.path.join("..", "src", "aalibrary", 'console', '*.py'))
console_tools_paths = [p for p in console_tools_paths if not p.endswith('__init__.py')]
console_tools = [os.path.basename(p)[:-3] for p in console_tools_paths]

In [3]:
# {tool_name: documentation} dictionary to store the documentation for each tool
tool_docs = {}
# Generate string for documentation for each console tool
documentation = "# Console Tools Documentation\n\n"

pbar = tqdm(console_tools, desc="Generating docs...")
for tool in pbar:
    pbar.set_description(f"Generating docs|{tool}")
    f = StringIO()
    with redirect_stdout(f):
        try:
            exec(f"from aalibrary.console.{tool} import print_help\nprint_help()")
        except Exception as e:
            pass
    output = f.getvalue()
    output = output.replace("    ", "\t").lstrip().rstrip("\t\n")
    # output = '\n'.join([line.lstrip("  ") for line in output.splitlines()])

    tool_docs[tool] = output
    documentation += f"## {tool}\n\n```bash\n{tool_docs[tool]}\n```\n\n"

Generating docs...:   0%|          | 0/41 [00:00<?, ?it/s]

Generating docs|aa_evl:   2%|▏         | 1/41 [00:10<06:43, 10.09s/it]             c:\Users\Hannah Khan\.conda\envs\aalibrary\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating docs|aa_fetch:  37%|███▋      | 15/41 [00:10<00:13,  1.91it/s]aa-fetch — Execute a YAML-driven multi-fetch job (no stdout output)

WHAT THIS TOOL DOES
  • Reads a YAML file that describes one or more fetch "requests"
  • Parses it using aalibrary.utils.multi_fetch_yaml_parser
  • Executes the fetch workflow (network / database / download work)
  • Creates a download directory for this run
  • Logs progress and errors to stderr (via loguru)
  • IMPORTANT: This tool DOES NOT print anything to stdout on success.

WHY THERE IS NO STDOUT
  aa-fetch is an "action" tool. It is designed to execute work and log to stderr.
  This keeps stdout

Generating docs|aa_sv:  73%|███████▎  | 30/41 [00:14<00:04,  2.68it/s]             

Generating docs|aa_ts: 100%|██████████| 41/41 [00:15<00:00,  2.73it/s]       


In [4]:
pprint(tool_docs)

{'aa_absorption': 'Usage: aa-absorption [OPTIONS]\n'
                  '\n'
                  '\tOptions:\n'
                  '\t  --frequency FLOAT_OR_LIST  Frequency in Hz (e.g., '
                  '38000) or comma-separated list (e.g., 38000,120000). '
                  'Required.\n'
                  '\t  --temperature FLOAT\t\t Temperature in °C. Default: 27\n'
                  '\t  --salinity FLOAT\t\t\tSalinity in PSU. Default: 35\n'
                  '\t  --pressure FLOAT\t\t\tPressure in dbar. Default: 10\n'
                  '\t  --pH FLOAT\t\t\t\t  pH of seawater. Default: 8.1\n'
                  "\t  --formula-source STR\t\tFormula source: 'AM', 'FG', or "
                  "'AZFP'. Default: AM\n"
                  '\t  -o, --output_path PATH\t  Optional NetCDF output path '
                  '(default: none).\n'
                  '\t  --quiet\t\t\t\t\t Print only numeric values (or '
                  'array).\n'
                  '\t  -h, --help\t\t\t\t  Show this hel

In [5]:
print(documentation)

# Console Tools Documentation

## aa_absorption

```bash
Usage: aa-absorption [OPTIONS]

	Options:
	  --frequency FLOAT_OR_LIST  Frequency in Hz (e.g., 38000) or comma-separated list (e.g., 38000,120000). Required.
	  --temperature FLOAT		 Temperature in °C. Default: 27
	  --salinity FLOAT			Salinity in PSU. Default: 35
	  --pressure FLOAT			Pressure in dbar. Default: 10
	  --pH FLOAT				  pH of seawater. Default: 8.1
	  --formula-source STR		Formula source: 'AM', 'FG', or 'AZFP'. Default: AM
	  -o, --output_path PATH	  Optional NetCDF output path (default: none).
	  --quiet					 Print only numeric values (or array).
	  -h, --help				  Show this help message and exit.

	Description:
	  Computes seawater absorption in dB/m for given frequency(ies) and parameters.
```

## aa_abundance

```bash
Usage: aa-abundance [OPTIONS] [INPUT_PATH]

	Arguments:
	  INPUT_PATH				   Path to a NetCDF file (.nc) containing a calibrated
								   Dataset with 'echo_range'. Optional; defaults to
								 

In [6]:
print(os.path.normpath(os.path.join(os.getcwd(), "..", "docs","documentation", "console_tools.md")))
docs_path = os.path.normpath(os.path.join(os.getcwd(), "..", "docs","documentation", "console_tools.md"))
with open(docs_path, "w+") as f:
    f.write(documentation)


c:\Users\Hannah Khan\Desktop\repos\AA-SI_aalibrary\docs\documentation\console_tools.md
